# Einsteinanium NNUE v1

Run on an A100 if Colab offers one; L4 is also sufficient. This notebook creates a new dataset and checkpoint path and deliberately refuses to overwrite an earlier run.

In [ ]:
!pip -q install 'datasets>=3.0,<5' python-chess
!git clone --branch codex/search-v2 \
  https://github.com/divitkashyap/aichessathon-starter.git \
  /content/Einsteinanium
%cd /content/Einsteinanium
!git log -1 --oneline
!python -m unittest discover -s tests -p 'test_nnue*.py'

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
RUN_ROOT = '/content/drive/MyDrive/einsteinanium/nnue-v1'
DATASET = f'{RUN_ROOT}/data'
CHECKPOINT = f'{RUN_ROOT}/einsteinanium-nnue-v1.pt'
WEIGHTS = f'{RUN_ROOT}/einsteinanium-nnue-v1.npz'

## Build 2 million deduplicated positions

The source has 958M evaluation rows, so we stream only what this first experiment needs. Mate labels are mapped near ±10,000 cp; all labels retain the source's side-to-move perspective.

In [ ]:
!python -m tools.stream_lichess_nnue \
  --output "$DATASET" \
  --positions 2000000 \
  --minimum-depth 18 \
  --shard-size 100000

## Train and export

Six epochs is the first gate, not a sacred hyperparameter. The trainer saves only the best validation checkpoint.

In [ ]:
!python -m tools.train_nnue \
  --dataset "$DATASET" \
  --output "$CHECKPOINT" \
  --epochs 6 \
  --batch-size 16384 \
  --feature-hidden 128

In [ ]:
!python -m tools.export_nnue "$CHECKPOINT" "$WEIGHTS"
!ls -lh "$CHECKPOINT" "$WEIGHTS"

When this finishes, download or share **einsteinanium-nnue-v1.npz**. Do not submit it directly: it still needs integration, inference-speed measurement, tactical regression tests, and a paired tournament against V4.